In [0]:
# ==============================================================================
# PySpark Data Engineering Pipeline - Milestone 2
# Dataset: Crypto Technical Price Data (Upbit 1m Top 3)
# Volume Path: /Volumes/workspace/default/cryptomilestone2/upbit_1m_top_3_technical_price.csv
# ==============================================================================

In [0]:
from pyspark.sql import SparkSession
from pyspark.sql.functions import col, lit, count, when, isnan
from pyspark.sql.types import (
    StructType, StructField, StringType, DoubleType, TimestampType, IntegerType
)
df_raw.show(5, truncate=False)

In [0]:
# Initialize Spark Session
spark = SparkSession.builder \
    .appName("Crypto_Milestone_2_ETL") \
    .getOrCreate()
    df_raw.printSchema()

In [0]:
# File path from unity catalog volume
input_path = "/Volumes/workspace/default/cryptomilestone2/upbit_1m_top_3_technical_price.csv"
output_path = "/Volumes/workspace/default/cryptomilestone2/processed_crypto_parquet"

In [0]:

# ------------------------------------------------------------------------------
# Task 3: Read Data with Appropriate Read Mode (PERMISSIVE)
# ------------------------------------------------------------------------------
# PERMISSIVE mode captures malformed records into a designated column `_corrupt_record`
# without stopping the execution flow.

In [0]:
df_raw = spark.read \
    .option("header", "true") \
    .option("inferSchema", "true") \
    .option("mode", "PERMISSIVE") \
    .option("columnNameOfCorruptRecord", "_corrupt_record") \
    .csv(input_path)

In [0]:
print("\n========================================")
print("STAGE 1: RAW DATA (df_raw)")
print("========================================")
print(f"Total Rows: {df_raw.count()}")
print(f"Total Columns: {len(df_raw.columns)}")
print("\nFirst 10 rows:")
df_raw.show(10, truncate=False)

In [0]:
print("\n========================================")
print("STAGE 2: AFTER SCHEMA VALIDATION (df)")
print("========================================")
print(f"Total Rows: {df.count()}")
print(f"Total Columns: {len(df.columns)}")
print("\nFirst 10 rows:")
df.show(10, truncate=False)

In [0]:
print("\n========================================")
print("STAGE 3: AFTER TRANSFORMATIONS (df_transformed)")
print("========================================")
print(f"Total Rows: {df_transformed.count()}")
print(f"Total Columns: {len(df_transformed.columns)}")
print("\nFirst 10 rows:")
df_transformed.show(10, truncate=False)

In [0]:
print("\n========================================")
print("STAGE 4: AFTER NULL HANDLING (df_clean)")
print("========================================")
print(f"Total Rows: {df_clean.count()}")
print(f"Total Columns: {len(df_clean.columns)}")
print("\nFirst 10 rows:")
df_clean.show(10, truncate=False)

In [0]:
print("\n========================================")
print("STAGE 5: FINAL DEDUPLICATED DATA (df_deduped)")
print("========================================")
print(f"Total Rows: {df_deduped.count()}")
print(f"Total Columns: {len(df_deduped.columns)}")
print("\nFirst 10 rows:")
df_deduped.show(10, truncate=False)

In [0]:

# ==============================================================================
# STEP 4: SCHEMA DEFINITION & RE-READING DATA
# ==============================================================================
print("--- [STEP 4] Explicit Schema Enforcement ---")
custom_schema = StructType([
    StructField("timestamp", TimestampType(), True),
    StructField("market", StringType(), True),
    StructField("opening_price", DoubleType(), True),
    StructField("high_price", DoubleType(), True),
    StructField("low_price", DoubleType(), True),
    StructField("trade_price", DoubleType(), True),
    StructField("candle_acc_trade_volume", DoubleType(), True),
    StructField("candle_acc_trade_price", DoubleType(), True),
    StructField("_corrupt_record", StringType(), True)
])

df = spark.read \
    .option("header", "true") \
    .option("mode", "PERMISSIVE") \
    .option("columnNameOfCorruptRecord", "_corrupt_record") \
    .schema(custom_schema) \
    .csv(input_path)

if "_corrupt_record" in df.columns:
    df = df.drop("_corrupt_record")

# CELL DISPLAY 3: Data After Explicit Schema Enforcement
print("\n>>> CELL DISPLAY 3: Data With Explicit Custom Schema Enforced")
df.show(5, truncate=False)

In [0]:
# ------------------------------------------------------------------------------
# Task 5: Corrupted Record Detection
# ------------------------------------------------------------------------------
print("\n=== TASK 5: CORRUPTED RECORD DETECTION ===")

if "_corrupt_record" in df_raw.columns:
    corrupt_df = df_raw.filter(col("_corrupt_record").isNotNull())
    corrupt_count = corrupt_df.count()
    print(f"Number of corrupt records found: {corrupt_count}")
    
    if corrupt_count > 0:
        corrupt_df.show(truncate=False)
    else:
        print("Documented: No corrupted records found in the dataset using PERMISSIVE mode.")

In [0]:
df_transformed.show(5, truncate=False)

In [0]:
# ------------------------------------------------------------------------------
# Task 6: Schema Validation and Creation
# ------------------------------------------------------------------------------
print("\n=== TASK 6: SCHEMA CREATION & RE-READING DATA ===")

In [0]:
# Defining an explicit schema for optimal performance and strict type enforcement
custom_schema = StructType([
    StructField("timestamp", TimestampType(), True),
    StructField("market", StringType(), True),
    StructField("opening_price", DoubleType(), True),
    StructField("high_price", DoubleType(), True),
    StructField("low_price", DoubleType(), True),
    StructField("trade_price", DoubleType(), True),
    StructField("candle_acc_trade_volume", DoubleType(), True),
    StructField("candle_acc_trade_price", DoubleType(), True),
    StructField("_corrupt_record", StringType(), True)
])

In [0]:
# ------------------------------------------------------------------------------
# Task 7: Data Transformations (5-7 Transformations Applied)
# ------------------------------------------------------------------------------
print("\n=== TASK 7: DATA TRANSFORMATIONS ===")

# Transformation 1: Filter rows where trade_price > 0 (Filter/Where)
df_transformed = df.filter(col("trade_price") > 0)

# Transformation 2: Add constant/metadata column (Literal)
df_transformed = df_transformed.withColumn("data_source", lit("Upbit_Exchange"))

# Transformation 3: Add calculated column - Candle Price Range (Adding Columns)
df_transformed = df_transformed.withColumn("candle_range", col("high_price") - col("low_price"))

# Transformation 4: Add calculated column - Spread Percentage (Adding Columns)
df_transformed = df_transformed.withColumn("spread_pct", ((col("high_price") - col("low_price")) / col("opening_price")) * 100)

# Transformation 5: Rename columns (Renaming Columns)
df_transformed = df_transformed \
    .withColumnRenamed("candle_acc_trade_volume", "total_volume") \
    .withColumnRenamed("candle_acc_trade_price", "total_trade_value")

# Transformation 6: Cast data type (Casting Data Types)
df_transformed = df_transformed.withColumn("candle_range", col("candle_range").cast(DoubleType()))

# Transformation 7: Select & Alias columns (Aliasing)
df_transformed = df_transformed.select(
    col("timestamp").alias("event_time"),
    col("market").alias("crypto_symbol"),
    col("opening_price").alias("open"),
    col("high_price").alias("high"),
    col("low_price").alias("low"),
    col("trade_price").alias("close"),
    col("total_volume"),
    col("total_trade_value"),
    col("candle_range"),
    col("spread_pct"),
    col("data_source")
)

df_transformed.show(5)

In [0]:
# ------------------------------------------------------------------------------
# Task 8: Null Value Handling
# ------------------------------------------------------------------------------
print("\n=== TASK 8: NULL VALUE HANDLING ===")

# 8.a Identify and print all null values across columns
# Only apply isnan() to numeric columns, not timestamp or string columns
numeric_cols = [c for c in df_transformed.columns if df_transformed.schema[c].dataType.typeName() in ['double', 'float']]
non_numeric_cols = [c for c in df_transformed.columns if c not in numeric_cols]

null_counts = df_transformed.select(
    [count(when(col(c).isNull() | isnan(c), c)).alias(c) for c in numeric_cols] +
    [count(when(col(c).isNull(), c)).alias(c) for c in non_numeric_cols]
)
print("8.a Null value count per column:")
null_counts.show()

# 8.b Handle null values (Removing null timestamp/symbol or imputing numeric zeros)
df_clean = df_transformed.dropna(subset=["event_time", "crypto_symbol"])
df_clean = df_clean.fillna(0, subset=["open", "high", "low", "close", "total_volume"])

In [0]:

# ------------------------------------------------------------------------------
# Task 9: Duplicate Removal
# ------------------------------------------------------------------------------
print("\n=== TASK 9: DUPLICATE REMOVAL ===")

initial_clean_count = df_clean.count()
df_deduped = df_clean.dropDuplicates(["event_time", "crypto_symbol"])
final_dedup_count = df_deduped.count()

print(f"Rows before deduplication: {initial_clean_count}")
print(f"Rows after deduplication: {final_dedup_count}")
print(f"Duplicates removed: {initial_clean_count - final_dedup_count}")

In [0]:
# ------------------------------------------------------------------------------
# Task 10: Write Processed Data
# ------------------------------------------------------------------------------
print("\n=== TASK 10: WRITING PROCESSED DATA ===")

# Save transformed data in Parquet format with Overwrite mode
df_deduped.write \
    .mode("overwrite") \
    .format("parquet") \
    .save(output_path)

# print(f"Data successfully saved in Parquet format to: {output_path}")